### Lexicon Builder


In [1]:
import os
import sys
import re
import json
import pickle

import requests
import spacy 
import nltk

from collections import Counter
import pandas as pd
from tqdm import tqdm

sys.path.append('..')

from utils.json import *
from utils.dataset import *

# TODO Implement manual labeling of languages
# - Alternatives to FastText
# - Manual Labeling
# - Not crucial but useful

tqdm.pandas()


In [2]:
# !python -m spacy download es_core_news_sm
# !python -m spacy download ca_core_news_sm

In [3]:
# Define the paths
path_annotations = "../data/annotations"
path_lexicons = "../data/lexicons"
path_negation = "../data/lexicons/negation"
path_uncertainty = "../data/lexicons/uncertainty"
path_model = "../data/lexicon/models"

In [4]:
"""
# -- FastText Language Detection (Not works as expected) --

# !pip install fasttext spacy
# !python -m spacy download es_core_news_sm
# !python -m spacy download ca_core_news_sm

# URL of the FastText model
url = "https://dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.bin"
 
# Path to fastText model
file_model = os.path.join(path_model, "lid.176.bin")

# Download fastText model if needed
if not os.path.exists(file_model):
    print("Downloading fastText language detection model...")
    url = "https://dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.bin"
    response = requests.get(url, stream=True)
    response.raise_for_status() # Raise an exception
    with open(file_model, "wb") as f:
        f.write(response.content)
    print("Model downloaded successfully!")
else:
    print("FastText model already exists at:", file_model)
"""

'\n# -- FastText Language Detection (Not works as expected) --\n\n# !pip install fasttext spacy\n# !python -m spacy download es_core_news_sm\n# !python -m spacy download ca_core_news_sm\n\n# URL of the FastText model\nurl = "https://dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.bin"\n \n# Path to fastText model\nfile_model = os.path.join(path_model, "lid.176.bin")\n\n# Download fastText model if needed\nif not os.path.exists(file_model):\n    print("Downloading fastText language detection model...")\n    url = "https://dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.bin"\n    response = requests.get(url, stream=True)\n    response.raise_for_status() # Raise an exception\n    with open(file_model, "wb") as f:\n        f.write(response.content)\n    print("Model downloaded successfully!")\nelse:\n    print("FastText model already exists at:", file_model)\n'

In [5]:
os.makedirs(path_negation, exist_ok=True)
os.makedirs(path_uncertainty, exist_ok=True)

In [6]:
# Load the training dataframe
with open(os.path.join(path_annotations, "df_train.pkl"), "rb") as f:
    df_train = pickle.load(f)

display(df_train.head())

print("Label distribution:")
print(df_train["label"].value_counts())

,doc_index,doc_id,result_id,word_idx,start,end,label,text,line_number
0,0,19026587,ent0,68,68,69,NEG,no,19026587_0
1,0,19026587,ent1,69,69,72,NSCO,habitos toxicos.,19026587_0
2,0,19026587,ent2,38,38,39,NSCO,cistoscopia,19026587_2
3,0,19026587,ent3,41,41,42,NEG,negativa,19026587_2
4,0,19026587,ent4,42,42,45,NSCO,para lesiones malignas,19026587_2


Label distribution:
label
NEG     4272
NSCO    4061
UNC      457
USCO     449
Name: count, dtype: int64


In [7]:
def clean_text(text):
    if not text:
        return ""
    
    text = text.lower().strip()
    text = re.sub(r"^[\.,;\s]+|[\.;,\s]+$", "", text) # Remove punctuation at beginning and end
    
    return text

In [8]:
neg_df = df_train[df_train["label"] == "NEG"].copy() # Filter only NEG labels
unc_df = df_train[df_train["label"] == "UNC"].copy() # Filter only UNC labels
print(f"Found {len(neg_df)} negation cues and {len(unc_df)} uncertainty cues")


# Obtain unique negation and uncertainty cues
unique_neg_cues = set(neg_df["text"].apply(clean_text).unique())
unique_unc_cues = set(unc_df["text"].apply(clean_text).unique())
print(f"Unique negation cues: {len(unique_neg_cues)}")
print(f"Unique uncertainty cues: {len(unique_unc_cues)}")

Found 4272 negation cues and 457 uncertainty cues
Unique negation cues: 60
Unique uncertainty cues: 79


In [9]:
# Clean and normalize terms
neg_df["clean_text"] = neg_df["text"].apply(clean_text)
unc_df["clean_text"] = unc_df["text"].apply(clean_text)

neg_df = neg_df[neg_df["clean_text"] != ""] # Remove empty terms
unc_df = unc_df[unc_df["clean_text"] != ""] # Remove empty terms
print(f"After cleaning: {len(neg_df)} negation cues and {len(unc_df)} uncertainty cues")

neg_counts = neg_df["clean_text"].value_counts().to_dict() # Count term frequencies
unc_counts = unc_df["clean_text"].value_counts().to_dict() # Count term frequencies


neg_lexicon = pd.DataFrame({ # NEG Lexicon DataFrames with unique terms
    "term": list(neg_counts.keys()),
    "freq": list(neg_counts.values())
})

unc_lexicon = pd.DataFrame({ # UNC Lexicon DataFrames with unique terms
    "term": list(unc_counts.keys()),
    "freq": list(unc_counts.values())
})


print("\nTOP negation cues:")
for term, count in sorted(neg_counts.items(), key=lambda x: x[1], reverse=True)[:]:
    print(f"{term}: {count}")

print("\nTOP uncertainty cues:")
for term, count in sorted(unc_counts.items(), key=lambda x: x[1], reverse=True)[:]:
    print(f"{term}: {count}")

After cleaning: 4272 negation cues and 457 uncertainty cues

TOP negation cues:
no: 1882
sin: 1420
negativo: 199
afebril: 188
niega: 136
negativos: 92
ausencia de: 64
negativa: 53
sense: 39
neg: 35
negativas: 24
asintomatica: 11
asintomatico: 10
ex: 9
descarta: 8
falta de: 7
inespecifico: 7
impide: 5
cede: 4
tampoco: 4
exfumador: 4
se retira: 4
nega: 4
ex-: 4
negativo): 3
negatividad de: 3
inespecificos: 3
descartada: 3
retiro: 3
inestabilidad: 3
negatiu: 3
retirar: 3
excepto: 2
desaparicion de: 2
desorientacion: 2
imposibilidad de: 2
incapacidad para: 2
ausencia: 2
niegan: 2
se desestimo: 1
ex fumador: 1
negaitvo: 1
suspendido: 1
indetectable: 1
negativa): 1
desorientado: 1
irregulares: 1
ceden: 1
rechaza: 1
se suspende: 1
negatividad: 1
imposibilidad: 1
desaparecen: 1
en ninguna: 1
ninguno: 1
desaparicion del: 1
atipicos: 1
arritmicos: 1
negatividad del: 1
inespecifico:: 1

TOP uncertainty cues:
compatible con: 58
probable: 54
sospecha de: 31
se orienta: 23
probablemente: 23
posible:

In [10]:
# (Continuing from the previous code block)
import nltk
from collections import defaultdict
from typing import Dict, Set, List, Union

# --- Functions from previous step (with modification to add_lemma) ---

def add_lemma(grammar_dict: Dict[str, Set[str]], input_word: str, target_lemma: str):
    """Adds lowercase input word mapping."""
    if not target_lemma or not input_word: return # Skip empty
    # Store input word in lowercase for case-insensitive lookup later
    grammar_dict[target_lemma].add(input_word.lower())

def convert_dict_to_cfg_string(grammar_dict: Dict[str, Set[str]]) -> str:
    """Converts dict to CFG string."""
    cfg_rules = []
    for target_lemma in sorted(grammar_dict.keys()):
        # Input words are already lowercase in the set
        input_words = sorted(list(grammar_dict[target_lemma]))
        # Use repr() for proper quoting in CFG string format
        productions = " | ".join(repr(word) for word in input_words)
        # Ensure target_lemma is valid (basic check)
        safe_target = target_lemma.replace('<','').replace('>','') # Use content if <> included
        if not safe_target.replace('_','').isalnum() or not safe_target[0].isalpha():
             print(f"Warning: Target '{target_lemma}' converted to '{safe_target}' may still be invalid Nonterminal.")
        cfg_rules.append(f"{safe_target} -> {productions}")
    return "\n".join(cfg_rules)

# --- Build the Grammar Dictionary (Example) ---

lemma_grammar_dict: Dict[str, Set[str]] = defaultdict(set)
print("Building grammar...")
# Add NEG examples (lowercase input word stored)
add_lemma(lemma_grammar_dict, input_word='negativo', target_lemma='_negativo')
add_lemma(lemma_grammar_dict, input_word='negativo.', target_lemma='_negativo')
add_lemma(lemma_grammar_dict, input_word='negativo)', target_lemma='_negativo')
add_lemma(lemma_grammar_dict, input_word='(negativo', target_lemma='_negativo')
add_lemma(lemma_grammar_dict, input_word='negativa', target_lemma='_negativo')
add_lemma(lemma_grammar_dict, input_word='negativa.', target_lemma='_negativo')
add_lemma(lemma_grammar_dict, input_word='negativa)', target_lemma='_negativo')
add_lemma(lemma_grammar_dict, input_word='(nagativa', target_lemma='_negativo')
add_lemma(lemma_grammar_dict, input_word='negativas', target_lemma='_negativo')
add_lemma(lemma_grammar_dict, input_word='negativos', target_lemma='_negativo')
add_lemma(lemma_grammar_dict, input_word='negaitvo', target_lemma='_negativo')
add_lemma(lemma_grammar_dict, input_word='negatiu', target_lemma='_negativo') # Catalan
add_lemma(lemma_grammar_dict, input_word='negatives', target_lemma='_negativo') # Catalan
add_lemma(lemma_grammar_dict, input_word='negatius', target_lemma='_negativo') # Catalan

add_lemma(lemma_grammar_dict, input_word='afebril', target_lemma='_afebril')
add_lemma(lemma_grammar_dict, input_word='afebril.', target_lemma='_afebril') # Added example with punctuation
add_lemma(lemma_grammar_dict, input_word='afebril,', target_lemma='_afebril') # Added example with punctuation


add_lemma(lemma_grammar_dict, input_word='sin', target_lemma='_sin')
add_lemma(lemma_grammar_dict, input_word='sense', target_lemma='_sin') # Catalan

add_lemma(lemma_grammar_dict, input_word='no', target_lemma='_no')

add_lemma(lemma_grammar_dict, input_word='ex', target_lemma='_ex')
add_lemma(lemma_grammar_dict, input_word='ex-', target_lemma='_ex')

# Add UNC examples
add_lemma(lemma_grammar_dict, input_word='probable', target_lemma='_probable')
add_lemma(lemma_grammar_dict, input_word='probables', target_lemma='_probable')
add_lemma(lemma_grammar_dict, input_word='posible', target_lemma='_posible')
add_lemma(lemma_grammar_dict, input_word='posibles', target_lemma='_posible')
add_lemma(lemma_grammar_dict, input_word='dudoso', target_lemma='_dudoso')
add_lemma(lemma_grammar_dict, input_word='dudosa', target_lemma='_dudoso')
add_lemma(lemma_grammar_dict, input_word='dudosos', target_lemma='_dudoso')

# Fumador
add_lemma(lemma_grammar_dict, input_word='exfumador', target_lemma='_exfumador')
add_lemma(lemma_grammar_dict, input_word='ex fumador', target_lemma='_exfumador')
add_lemma(lemma_grammar_dict, input_word='exfumadora', target_lemma='_exfumador')
add_lemma(lemma_grammar_dict, input_word='ex fumadora', target_lemma='_exfumador')
add_lemma(lemma_grammar_dict, input_word='ex-fumador', target_lemma='_exfumador')
add_lemma(lemma_grammar_dict, input_word='ex-fumadora', target_lemma='_exfumador')

add_lemma(lemma_grammar_dict, input_word='asintomatico', target_lemma='_asintomatico')
add_lemma(lemma_grammar_dict, input_word='asintomatica', target_lemma='_asintomatico')
add_lemma(lemma_grammar_dict, input_word='asintomaticos', target_lemma='_asintomatico')
add_lemma(lemma_grammar_dict, input_word='asintomaticas', target_lemma='_asintomatico')
add_lemma(lemma_grammar_dict, input_word='asintomatic', target_lemma='_asintomatico')

add_lemma(lemma_grammar_dict, input_word='inespecifico', target_lemma='_inespecifico')
add_lemma(lemma_grammar_dict, input_word='inespecifica', target_lemma='_inespecifico')
add_lemma(lemma_grammar_dict, input_word='inespecificos', target_lemma='_inespecifico')
add_lemma(lemma_grammar_dict, input_word='inespecificas', target_lemma='_inespecifico')
add_lemma(lemma_grammar_dict, input_word='inespecic', target_lemma='_inespecifico')

add_lemma(lemma_grammar_dict, input_word='inactivo', target_lemma='_inactivo')
add_lemma(lemma_grammar_dict, input_word='inactiva', target_lemma='_inactivo')
add_lemma(lemma_grammar_dict, input_word='inactivos', target_lemma='_inactivo')
add_lemma(lemma_grammar_dict, input_word='inactivas', target_lemma='_inactivo')
add_lemma(lemma_grammar_dict, input_word='inactiu', target_lemma='_inactivo')
add_lemma(lemma_grammar_dict, input_word='inactiva', target_lemma='_inactivo')
add_lemma(lemma_grammar_dict, input_word='inactius', target_lemma='_inactivo')
add_lemma(lemma_grammar_dict, input_word='inactives', target_lemma='_inactivo')

add_lemma(lemma_grammar_dict, input_word='del', target_lemma='_de')
add_lemma(lemma_grammar_dict, input_word='de', target_lemma='_de')
add_lemma(lemma_grammar_dict, input_word='d\'', target_lemma='_de')


# --- Create CFG String and NLTK Grammar Object ---
cfg_string = convert_dict_to_cfg_string(lemma_grammar_dict)
print("\nGenerated CFG String:")
print(cfg_string)

try:
    # Create the grammar object
    # Note: This grammar is purely lexical and lacks a start symbol / higher rules
    # needed for actual sentence parsing. We use it only for its productions.
    grammar = nltk.CFG.fromstring(cfg_string)
    print("\nNLTK CFG object created successfully.")
except ValueError as e:
    print(f"\nError creating NLTK CFG object: {e}")
    grammar = None # Set grammar to None if creation fails

# --- Build Reverse Map for Lookup ---

word_to_lemma_map: Dict[str, str] = {}
if grammar:
    for production in grammar.productions():
        if production.is_lexical():
            # rhs() is a tuple, get the single word terminal
            word = production.rhs()[0]
            # lhs() is the Nonterminal object, get its symbol string
            lemma = production.lhs().symbol()
            # Word should already be lowercase from modified add_lemma
            if isinstance(word, str): # Ensure it's a string terminal
                 word_to_lemma_map[word] = lemma

print(f"\nCreated word-to-lemma map with {len(word_to_lemma_map)} entries.")

def map_tokens_to_lemmas(tokens: List[str], lemma_map: Dict[str, str]) -> List[str]:
    """Maps a list of tokens to their corresponding lemmas using the provided map."""
    mapped_output = []
    for token in tokens:
        # Lookup lowercase token in the map
        lemma = lemma_map.get(token.lower(), token) # Default to original token if not found
        mapped_output.append(lemma)
    return mapped_output

# save lemmatizer
with open(os.path.join(path_lexicons, "lemma_grammar_dict.pkl"), "wb") as f:
    pickle.dump(lemma_grammar_dict, f)


Building grammar...

Generated CFG String:
_afebril -> 'afebril' | 'afebril,' | 'afebril.'
_asintomatico -> 'asintomatic' | 'asintomatica' | 'asintomaticas' | 'asintomatico' | 'asintomaticos'
_de -> "d'" | 'de' | 'del'
_dudoso -> 'dudosa' | 'dudoso' | 'dudosos'
_ex -> 'ex' | 'ex-'
_exfumador -> 'ex fumador' | 'ex fumadora' | 'ex-fumador' | 'ex-fumadora' | 'exfumador' | 'exfumadora'
_inactivo -> 'inactiu' | 'inactius' | 'inactiva' | 'inactivas' | 'inactives' | 'inactivo' | 'inactivos'
_inespecifico -> 'inespecic' | 'inespecifica' | 'inespecificas' | 'inespecifico' | 'inespecificos'
_negativo -> '(nagativa' | '(negativo' | 'negaitvo' | 'negatiu' | 'negatius' | 'negativa' | 'negativa)' | 'negativa.' | 'negativas' | 'negatives' | 'negativo' | 'negativo)' | 'negativo.' | 'negativos'
_no -> 'no'
_posible -> 'posible' | 'posibles'
_probable -> 'probable' | 'probables'
_sin -> 'sense' | 'sin'

NLTK CFG object created successfully.

Created word-to-lemma map with 55 entries.


In [11]:
import nltk
from typing import List, Dict

# Assume 'grammar' is the nltk.CFG object created successfully in the previous steps
# If not, you need to run the code that builds the lemma_grammar_dict,
# converts it to cfg_string, and then calls nltk.CFG.fromstring(cfg_string)

# --- Simple Parsing (Mapping) Function ---

def parse_tokens_with_lexical_grammar(grammar: nltk.CFG, sentence_tokens: list[str]) -> list[str]:
    """
    Applies a purely lexical NLTK grammar to map tokens in a sentence.

    This function iterates through the grammar's lexical rules (LEMMA -> 'word')
    to build a word-to-lemma lookup map. It then iterates through the input
    tokens, replacing any known token (case-insensitive) with its corresponding
    lemma symbol from the grammar. Tokens not found in the grammar are
    returned unchanged.

    This is NOT syntactic parsing but rather a form of lexical normalization or tagging.

    Args:
        grammar: An nltk.CFG object containing primarily lexical rules.
                 It must have been successfully created (not None).
        sentence_tokens: A list of strings representing the tokenized sentence.

    Returns:
        A list of strings where known tokens are replaced by their lemma
        symbols from the grammar. Returns the original list if grammar is
        invalid or sentence is empty.
    """
    if not isinstance(grammar, nltk.CFG) or not sentence_tokens:
        return sentence_tokens # Return original if grammar invalid or no tokens

    word_to_lemma_map: dict[str, str] = {}
    try:
        for production in grammar.productions():
            # Check if it's a lexical rule like: LEMMA -> 'word'
            if production.is_lexical() and isinstance(production.rhs()[0], str):
                word = production.rhs()[0] # The terminal word from CFG (should be lowercase)
                lemma = production.lhs().symbol() # The non-terminal lemma string (e.g., '_negativo')
                word_to_lemma_map[word] = lemma
    except Exception as e:
        print(f"Error processing grammar productions: {e}")
        return sentence_tokens # Return original tokens on error

    if not word_to_lemma_map:
        print("Warning: No lexical rules found in the grammar to build a map.")
        # Fall through to map_tokens_to_lemmas, which will just return original tokens

    # 2. Map input tokens using the created map
    mapped_output = []
    for token in sentence_tokens:
        # Lookup the lowercase version of the token in the map
        # If not found, default to the original token itself
        lemma = word_to_lemma_map.get(token.lower(), token)
        mapped_output.append(lemma)

    return mapped_output

# --- Example Usage ---
# (Requires the 'grammar' object from the previous code block to be valid)

# Example token lists (pre-tokenized)
text1 = "el paciente esta afebril."

# Tokenize the text
tokens1 = nltk.word_tokenize(text1)
print("Original Tokens:", tokens1)

# Apply the lexical grammar mapping
parsed_tokens1 = parse_tokens_with_lexical_grammar(grammar, tokens1)
print("Parsed Tokens:", parsed_tokens1)

Original Tokens: ['el', 'paciente', 'esta', 'afebril', '.']
Parsed Tokens: ['el', 'paciente', 'esta', '_afebril', '.']


In [12]:
# create dataframe with the lemmatized terms
neg_lexicon["lemma"] = neg_lexicon["term"].progress_apply(lambda x: map_tokens_to_lemmas([x], word_to_lemma_map)[0])
unc_lexicon["lemma"] = unc_lexicon["term"].progress_apply(lambda x: map_tokens_to_lemmas([x], word_to_lemma_map)[0])

100%|██████████| 79/79 [00:00<00:00, 36890.45it/s]


In [13]:
neg_lexicon

,term,freq,lemma
0,no,1882,_no
1,sin,1420,_sin
2,negativo,199,_negativo
3,afebril,188,_afebril
4,niega,136,niega
5,negativos,92,_negativo
6,ausencia de,64,ausencia de
7,negativa,53,_negativo
8,sense,39,_sin
9,neg,35,neg


We need to remove some duplicates ('no' and 'sin'):

In [14]:
unc_lexicon.drop_duplicates(subset=["lemma"], inplace=True)
neg_lexicon.drop_duplicates(subset=["lemma"], inplace=True)

# take out "_no" and "_sin" from the uncertainty lexicon
unc_lexicon = unc_lexicon[~unc_lexicon["lemma"].isin(["_no", "_sin"])]
unc_lexicon

,term,freq,lemma
0,compatible con,58,compatible con
1,probable,54,_probable
2,sospecha de,31,sospecha de
3,se orienta,23,se orienta
4,probablemente,23,probablemente
...,...,...,...
74,indeterminado,1,indeterminado
75,no clara,1,no clara
76,clara,1,clara
77,orientan como,1,orientan como


In [15]:
# Save to negation and uncertainty directories
neg_lexicon.to_csv(os.path.join(path_negation, "negation.csv"), index=False)
unc_lexicon.to_csv(os.path.join(path_uncertainty, "uncertainty.csv"), index=False)

print(f"Saved {len(neg_lexicon)} Spanish and {len(neg_lexicon)} Catalan negation terms")
print(f"Saved {len(unc_lexicon)} Spanish and {len(unc_lexicon)} Catalan uncertainty terms")

Saved 48 Spanish and 48 Catalan negation terms
Saved 73 Spanish and 73 Catalan uncertainty terms
